# Amaliyot 6. kNN

6-interaktiv laboratoriyaga («Eng yaqin k ta qoʻshni usuli») hamroh. Kurs mavzusi: eng yaqin k ta qoʻshni usuli bilan tasniflash

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.5, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Kross-validatsiya bilan k ni tanlash

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, train_test_split

vec = TfidfVectorizer()
X = vec.fit_transform(df.text); y = df.label.values
ks = range(1, 31)
sc = [cross_val_score(KNeighborsClassifier(k, metric="cosine", algorithm="brute"), X, y, cv=5).mean() for k in ks]
plt.plot(ks, sc, marker="o"); plt.xlabel("k"); plt.ylabel("accuracy (CV)"); plt.title("kNN: toʻgʻrilik va qoʻshnilar soni"); plt.show()
print("Eng yaxshi k:", ks[int(np.argmax(sc))], round(max(sc), 3))

## 2. Metrika va ovoz ogʻirligi

In [ ]:
rows = []
for metric in ["cosine", "euclidean"]:
    for w in ["uniform", "distance"]:
        m = KNeighborsClassifier(7, metric=metric, weights=w, algorithm="brute")
        rows.append((metric, w, cross_val_score(m, X, y, cv=5).mean()))
pd.DataFrame(rows, columns=["metrika", "ovozlar", "accuracy"]).round(3)

## 3. Qoʻshnilar kim: kNN ning tushuntirilishi

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(df.text, df.label, test_size=0.3, random_state=0)
vec2 = TfidfVectorizer().fit(Xtr)
knn = KNeighborsClassifier(5, metric="cosine", algorithm="brute").fit(vec2.transform(Xtr), ytr)
query = "jamoa gol oʻyin byudjet qonun"
dist, idx = knn.kneighbors(vec2.transform([query]), 5)
print("Soʻrov:", query, "->", knn.predict(vec2.transform([query]))[0])
for d, i in zip(dist[0], idx[0]):
    print(f"oʻxshashlik {1-d:.2f} | {ytr.iloc[i]:11s} | {Xtr.iloc[i]}")

## Topshiriqlar

1. Oʻquv maʼlumotlarida k boʻyicha toʻgʻrilik grafigini quring (kross-validatsiyasiz). Nima uchun k=1 da toʻgʻrilik 100%?
2. Korpus oʻsganda (n_per_class=1500) kNN va logistik regressiyaning bashorat vaqtini solishtiring. Bu «oʻqitish» yoʻqligi haqida nimani aytadi?
3. TF-IDF oʻrniga TF (CountVectorizer) ni sinab koʻring. Toʻgʻrilik qanday oʻzgaradi?